In [ ]:
%matplotlib inline
import time
import matplotlib.pyplot as plt
import gc
from functools import reduce

# ==========================================
# Tool function：Extended Euclidean Algorithm
# ==========================================
def xgcd_custom(a, b):
    x0, x1, y0, y1 = 1, 0, 0, 1
    while b != 0:
        q, a, b = a // b, b, a % b
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return a, x0, y0

# ==========================================
# [baseline]  SNF (Smith Normal Form)-based solver
# ==========================================
def solve_traditional_snf_sage(A_coeff, B_const, M_moduli):
    n = len(M_moduli)
    M_total = prod(M_moduli)
    
    K = Matrix(ZZ, n, n)
    b_prime = vector(ZZ, n)
    for i in range(n):
        M_i = M_total // M_moduli[i]
        K[i, :] = M_i * vector(ZZ, A_coeff[i])
        b_prime[i] = M_i * B_const[i]

    D_snf, P_snf, Q_snf = K.smith_form()

    b_double_prime = P_snf * b_prime
    t = vector(ZZ, n)
    h_sols = []
    periods = []
    
    for i in range(n):
        d_i = D_snf[i, i]
        g_i = gcd(d_i, M_total) if d_i != 0 else M_total
        
        if b_double_prime[i] % g_i != 0:
            raise ValueError(f"SNF 判定无解! d_i={d_i}")
            
        d_prime = d_i // g_i
        b_prime_val = b_double_prime[i] // g_i
        m_prime = M_total // g_i
        
        y0 = (b_prime_val * inverse_mod(d_prime, m_prime)) % m_prime if m_prime > 1 else 0
        t[i] = y0
        
        if g_i > 1:
            step = m_prime
            basis_y = vector(ZZ, n)
            basis_y[i] = step
            basis_x = (Q_snf * basis_y)
            basis_x_mod = [int(x % M_total) for x in basis_x]
            h_sols.append(tuple(basis_x_mod))
            periods.append(int(g_i))

    s_sol = (Q_snf * t)
    s_sol_mod = [int(x % M_total) for x in s_sol]
    return tuple(s_sol_mod), h_sols, periods
# ==========================================
# Our method: HNF-based solver
# ==========================================
def solve_method_5_truncated_sage(A_coeff, B_const, M_moduli):
    n = len(M_moduli)
    M_total = prod(M_moduli)
    
    # 1. CRT-Compress (O(n^2))
    K_vec = [0] * n
    B_val = 0
    for i in range(n):
        M_i = M_total // M_moduli[i]
        for j in range(n):
            K_vec[j] = (K_vec[j] + M_i * A_coeff[i][j]) % M_total
        B_val = (B_val + M_i * B_const[i]) % M_total
        

    
    # 2. GCD Sequence and Bézout Coefficient Matrix
    g_arr = [0] * (n + 1)
    g_arr[n] = M_total
    C_arr = [[0] * n for _ in range(n)]

    for i in range(n - 1, -1, -1):
        d, a, b = xgcd_custom(K_vec[i], g_arr[i + 1])
        g_arr[i] = d
        C_arr[i][i] = a % M_total
        for j in range(i + 1, n):
            C_arr[i][j] = (b * C_arr[i + 1][j]) % M_total

    if g_arr[0] != 1:
        raise ValueError("全局 GCD 不为 1，方程不一定有解或无法生成特解")
            
    # 3. Specific solution s
    s_sol = [0] * n
    for i in range(n):
        s_sol[i] = (B_val * C_arr[0][i]) % M_total
        
    h_sols = []; periods = []
    
    # 4. Diagonal entries and Off-diagonal entries（no Standard normalization）
    #   Output:Upper triangular lattice basis matrix H ，K·H[i] ≡ 0 (mod M_total)
    h_sols = []
    periods = []
    H = [[0] * n for _ in range(n)]

    for i in range(n - 1, -1, -1):
        H[i][i] = g_arr[i + 1] // g_arr[i]
        multiplier = K_vec[i] // g_arr[i]
        for j in range(i + 1, n):
            H[i][j] = (-multiplier * C_arr[i + 1][j]) % M_total

        # Period of H[i]
        vec = H[i]
        g_all = M_total
        for val in vec:
            g_all = gcd(g_all, val)
        period = M_total // g_all

        if period > 1:
            h_sols.append(tuple(int(x) for x in vec))
            periods.append(int(period))

    # Output:Specific solution s、lattice basis matrix H 、Periods、and n
    return tuple(int(x) for x in s_sol), h_sols, periods, n

# ==========================================
# Verify random solutions
# ==========================================
def generate_and_verify_random_solution(A_coeff, B_const, M_moduli, M_total, s_sol, h_sols, periods):
    n = len(M_moduli)
    t_coeffs = [ZZ.random_element(0, p) for p in periods]
    x_rand = vector(ZZ, s_sol)
    for t, h in zip(t_coeffs, h_sols):
        x_rand += t * vector(ZZ, h)
    x_rand_mod = [int(val % M_total) for val in x_rand]
    for i in range(n):
        val = sum(A_coeff[i][j] * x_rand_mod[j] for j in range(n))
        if (val - B_const[i]) % M_moduli[i] != 0:
            return False 
    return True

# ==========================================
# Benchmark
# ==========================================
def run_benchmark_sage(n_start, n_end, n_step, run_snf=True, run_m5=True):
    header = f"{'Size (n)':<8}"
    if run_snf: header += f" | {'Trad. SNF Time':<18} | {'SNF Valid':<10}"
    if run_m5: header += f" | {'Method 5 Time':<18} | {'M5 Valid':<8}"
    print(header)
    print("-" * len(header))
    
    sizes, t_snf, t_m5 = [], [], []
    prime_list = list(Primes()[:2*n_end])
    
    for n in range(n_start, n_end + 1, n_step):
        m_moduli = prime_list[:n]
        M_total = prod(m_moduli)
        
        # Generate random and solvable MLCs
        while True:
            A_mat = random_matrix(ZZ, n, n, x=-1000, y=1001)
            A_coeff = [list(row) for row in A_mat.rows()]
            K_test = [0] * n
            for i in range(n):
                Mi = M_total // m_moduli[i]
                for j in range(n):
                    K_test[j] = (K_test[j] + Mi * A_coeff[i][j]) % M_total
            if gcd(K_test + [M_total]) == 1:
                break
                
        X_true = [ZZ.random_element(-1000, 1000) for _ in range(n)]
        B_const = [sum(A_coeff[i][j] * X_true[j] for j in range(n)) % m_moduli[i] for i in range(n)]
        
        row_str = f"{n:<8}"
        sizes.append(n)
        
        # --- 1. Run SNF-based solver ---
        if run_snf:
            try:
                start = time.perf_counter()
                s_snf, h_snf, p_snf = solve_traditional_snf_sage(A_coeff, B_const, m_moduli)
                dt_snf = time.perf_counter() - start
                v_snf = generate_and_verify_random_solution(A_coeff, B_const, m_moduli, M_total, s_snf, h_snf, p_snf)
                t_snf.append(dt_snf)
                row_str += f" | {dt_snf:<16.5f} s | {'Yes' if v_snf else 'No':<10}"
            except Exception as e:
                t_snf.append(None)
                row_str += f" | {'Error/Timeout':<16}   | {'N/A':<10}"
                
        # --- 2. Run HNF-based solver ---
        if run_m5:
            start = time.perf_counter()
            s5, h5, p5, _ = solve_method_5_truncated_sage(A_coeff, B_const, m_moduli)
            dt5 = time.perf_counter() - start
            v5 = generate_and_verify_random_solution(A_coeff, B_const, m_moduli, M_total, s5, h5, p5)
            t_m5.append(dt5)
            row_str += f" | {dt5:<16.5f} s  | {'Yes' if v5 else 'No':<8}"
        
        print(row_str)
        gc.collect()

    plt.figure(figsize=(10, 6), dpi=300)
    
    if run_snf and any(x is not None for x in t_snf):
        valid_sizes = [s for s, t in zip(sizes, t_snf) if t is not None]
        valid_times = [t for t in t_snf if t is not None]
        plt.plot(valid_sizes, valid_times, marker='s', linestyle='--', color='crimson', linewidth=2.5, label='SNF')
        
    if run_m5:
        plt.plot(sizes, t_m5, marker='o', linestyle='-', color='forestgreen', linewidth=2.5, label='HNF')
    
    plt.xlabel('Matrix Order $n$', fontsize=18)
    plt.ylabel('Execution Time (Seconds)', fontsize=18)

    
    # plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend(fontsize=18, loc='upper left')
    plt.tick_params(axis='both', labelsize=16)
    plt.tight_layout()
    plt.show()

# ==========================================
# Options
# ==========================================

RUN_SNF = True  
RUN_M5  = True 

N_START = 5
N_END   = 50   
N_STEP  = 5

run_benchmark_sage(N_START, N_END, N_STEP, run_snf=RUN_SNF, run_m5=RUN_M5)

In [ ]:
%matplotlib inline
import time
import matplotlib.pyplot as plt
import gc
from functools import reduce

# ==========================================
# Tool function：Extended Euclidean Algorithm
# ==========================================
def xgcd_custom(a, b):
    x0, x1, y0, y1 = 1, 0, 0, 1
    while b != 0:
        q, a, b = a // b, b, a % b
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return a, x0, y0

# ==========================================
# Our method: HNF-based solver
# ==========================================
def solve_method_5_truncated_sage(A_coeff, B_const, M_moduli):
    n = len(M_moduli)
    M_total = prod(M_moduli)
    
    # 1. CRT-Compress (O(n^2))
    K_vec = [0] * n
    B_val = 0
    for i in range(n):
        M_i = M_total // M_moduli[i]
        for j in range(n):
            K_vec[j] = (K_vec[j] + M_i * A_coeff[i][j]) % M_total
        B_val = (B_val + M_i * B_const[i]) % M_total
        

    
    # 2. GCD Sequence and Bézout Coefficient Matrix
    g_arr = [0] * (n + 1)
    g_arr[n] = M_total
    C_arr = [[0] * n for _ in range(n)]

    for i in range(n - 1, -1, -1):
        d, a, b = xgcd_custom(K_vec[i], g_arr[i + 1])
        g_arr[i] = d
        C_arr[i][i] = a % M_total
        for j in range(i + 1, n):
            C_arr[i][j] = (b * C_arr[i + 1][j]) % M_total

    if g_arr[0] != 1:
        raise ValueError("全局 GCD 不为 1，方程不一定有解或无法生成特解")
            
    # 3. Specific solution s
    s_sol = [0] * n
    for i in range(n):
        s_sol[i] = (B_val * C_arr[0][i]) % M_total
        
    h_sols = []; periods = []
    
    # 4. Diagonal entries and Off-diagonal entries（no Standard normalization）
    #   Output:Upper triangular lattice basis matrix H ，K·H[i] ≡ 0 (mod M_total)
    h_sols = []
    periods = []
    H = [[0] * n for _ in range(n)]

    for i in range(n - 1, -1, -1):
        H[i][i] = g_arr[i + 1] // g_arr[i]
        multiplier = K_vec[i] // g_arr[i]
        for j in range(i + 1, n):
            H[i][j] = (-multiplier * C_arr[i + 1][j]) % M_total

        # Period of H[i]
        vec = H[i]
        g_all = M_total
        for val in vec:
            g_all = gcd(g_all, val)
        period = M_total // g_all

        if period > 1:
            h_sols.append(tuple(int(x) for x in vec))
            periods.append(int(period))

    # Output:Specific solution s、lattice basis matrix H 、Periods、and n
    return tuple(int(x) for x in s_sol), h_sols, periods, n
# ==========================================
# Verify random solutions
# ==========================================
def generate_and_verify_random_solution(A_coeff, B_const, M_moduli, M_total, s_sol, h_sols, periods):
    n = len(M_moduli)
    t_coeffs = [ZZ.random_element(0, p) for p in periods]
    x_rand = vector(ZZ, s_sol)
    for t, h in zip(t_coeffs, h_sols):
        x_rand += t * vector(ZZ, h)
    x_rand_mod = [int(val % M_total) for val in x_rand]
    for i in range(n):
        val = sum(A_coeff[i][j] * x_rand_mod[j] for j in range(n))
        if (val - B_const[i]) % M_moduli[i] != 0:
            return False 
    return True

# ==========================================
# Benchmark
# ==========================================
def run_m5_average_benchmark(n_start, n_end, n_step, iterations=10):
    print(f" 开始 Method 5 测试：n=[{n_start}, {n_end}], 每个规模重复 {iterations} 次取平均")
    print("=" * 70)
    print(f"{'Size (n)':<8} | {'Avg Time (s)':<15} | {'Min Time (s)':<15} | {'Verified?'}")
    print("-" * 70)

    sizes, avg_times = [], []
    prime_list = list(Primes()[:n_end])

    for n in range(n_start, n_end + 1, n_step):
        m_moduli = prime_list[:n]
        M_total = prod(m_moduli)

        times_n = []
        verified_ok = True

        for trial in range(iterations):
            # Generate random and solvable MLCs
            while True:
                A_mat = random_matrix(ZZ, n, n, x=-1000, y=1001)
                A_coeff = [list(row) for row in A_mat.rows()]
                K_test = [0] * n
                for i in range(n):
                    Mi = M_total // m_moduli[i]
                    for j in range(n):
                        K_test[j] = (K_test[j] + Mi * A_coeff[i][j]) % M_total
                if gcd(K_test + [M_total]) == 1:
                    break

            X_true = [ZZ.random_element(-100, 100) for _ in range(n)]
            B_const = [
                sum(A_coeff[i][j] * X_true[j] for j in range(n)) % m_moduli[i]
                for i in range(n)
            ]

            start = time.perf_counter()
            s5, h5, p5, _ = solve_method_5_truncated_sage(A_coeff, B_const, m_moduli)
            dt5 = time.perf_counter() - start

            times_n.append(dt5)

            if trial == 0:
                is_valid = generate_and_verify_random_solution(
                    A_coeff, B_const, m_moduli, M_total, s5, h5, p5
                )
                if not is_valid:
                    verified_ok = False

        avg_t = float(sum(times_n) / iterations)
        min_t = float(min(times_n))

        sizes.append(n)
        avg_times.append(avg_t)

        print(f"{n:<8} | {avg_t:<15.5f} | {min_t:<15.5f} | {'Yes' if verified_ok else 'No':<8}")
        gc.collect()

    plt.style.use('seaborn-v0_8-whitegrid')
    plt.figure(figsize=(10, 6), dpi=300)

    plt.plot(
        sizes, avg_times,
        marker='s', linestyle='-', color='forestgreen',
        linewidth=2.5, markersize=8,
        label='Mean Time'
    )

    plt.xlabel('Matrix Order $n$', fontsize=18)
    plt.ylabel('Execution Time (Seconds)', fontsize=18)
    plt.legend(fontsize=18, loc='upper left')
    plt.grid(False)
    plt.tick_params(axis='both', labelsize=16)
    plt.tight_layout()
    plt.show()
# ==========================================
# Options
# ==========================================
N_START = 50
N_END   = 500   
N_STEP  = 50
ITERATIONS = 10 # number of iterations for each dimension N
# ========================================
run_m5_average_benchmark(N_START, N_END, N_STEP, ITERATIONS)